In [1]:
# ============================================================
# STEP 1: CONNECT GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount('/content/drive')

print("Google Drive connected successfully.")

Mounted at /content/drive
Google Drive connected successfully.


In [2]:
# ============================================================
# STEP 2: INSTALL AND IMPORT REQUIRED LIBRARIES
# ============================================================

# SentencePiece is used to train our BPE subword tokenizers.
!pip install -q sentencepiece

import os
import pandas as pd
import sentencepiece as spm

print("Libraries imported successfully.")

ERROR: Operation cancelled by user
Libraries imported successfully.


In [6]:
# ============================================================
# STEP 3: DEFINE PROJECT PATHS
# ============================================================

PROJECT_ROOT = "/content/drive/MyDrive/NNDL_English_Hindi_Translator"

PROCESSED_DATA_DIR = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed"
)

TOKENIZER_DIR = os.path.join(
    PROJECT_ROOT,
    "tokenizers"
)

# Use the NEW 1M cleaned training dataset
TRAIN_PATH = os.path.join(
    PROCESSED_DATA_DIR,
    "train_clean_1m.parquet"
)

print("Training data:")
print(TRAIN_PATH)

print("\nTokenizer files will be saved in:")
print(TOKENIZER_DIR)

Training data:
/content/drive/MyDrive/NNDL_English_Hindi_Translator/data/processed/train_clean_1m.parquet

Tokenizer files will be saved in:
/content/drive/MyDrive/NNDL_English_Hindi_Translator/tokenizers


In [7]:
# ============================================================
# STEP 4: LOAD CLEANED TRAINING DATA
# ============================================================

train_df = pd.read_parquet(
    TRAIN_PATH
)

print("Training data loaded successfully.")
print("Shape:", train_df.shape)
print("Columns:", train_df.columns.tolist())

Training data loaded successfully.
Shape: (1000000, 4)
Columns: ['english', 'hindi', 'english_word_count', 'hindi_word_count']


In [8]:
# ============================================================
# STEP 5: CREATE 1M TEXT CORPORA FOR SENTENCEPIECE
# ============================================================
# These corpora are created from the new 1,000,000-pair
# cleaned training dataset.
#
# We use new filenames so the old 300K tokenizer setup
# remains preserved as backup.
# ============================================================

ENGLISH_CORPUS_PATH = os.path.join(
    TOKENIZER_DIR,
    "english_training_corpus_1m.txt"
)

HINDI_CORPUS_PATH = os.path.join(
    TOKENIZER_DIR,
    "hindi_training_corpus_1m.txt"
)

# Save one English sentence per line
train_df["english"].to_csv(
    ENGLISH_CORPUS_PATH,
    index=False,
    header=False
)

# Save one Hindi sentence per line
train_df["hindi"].to_csv(
    HINDI_CORPUS_PATH,
    index=False,
    header=False
)

print("1M tokenizer training corpora created successfully.")

print("\nEnglish corpus:")
print(ENGLISH_CORPUS_PATH)

print("\nHindi corpus:")
print(HINDI_CORPUS_PATH)

1M tokenizer training corpora created successfully.

English corpus:
/content/drive/MyDrive/NNDL_English_Hindi_Translator/tokenizers/english_training_corpus_1m.txt

Hindi corpus:
/content/drive/MyDrive/NNDL_English_Hindi_Translator/tokenizers/hindi_training_corpus_1m.txt


In [10]:
# ============================================================
# TOKENIZER CONFIGURATION
# ============================================================

VOCAB_SIZE = 16000

PAD_ID = 0
UNK_ID = 1
BOS_ID = 2
EOS_ID = 3

print("Tokenizer configuration loaded.")
print("Vocabulary size:", VOCAB_SIZE)

Tokenizer configuration loaded.
Vocabulary size: 16000


In [11]:
# ============================================================
# STEP 7: TRAIN ENGLISH SENTENCEPIECE BPE TOKENIZER - 1M DATA
# ============================================================

ENGLISH_MODEL_PREFIX = os.path.join(
    TOKENIZER_DIR,
    "english_bpe_1m"
)

spm.SentencePieceTrainer.train(
    input=ENGLISH_CORPUS_PATH,
    model_prefix=ENGLISH_MODEL_PREFIX,

    # BPE tokenizer
    model_type="bpe",

    # Keep the same vocabulary size for compatibility
    vocab_size=VOCAB_SIZE,

    character_coverage=1.0,

    # Fixed special token IDs
    pad_id=PAD_ID,
    unk_id=UNK_ID,
    bos_id=BOS_ID,
    eos_id=EOS_ID,

    pad_piece="<pad>",
    unk_piece="<unk>",
    bos_piece="<s>",
    eos_piece="</s>"
)

print("1M English BPE tokenizer trained successfully.")
print(
    "Saved model:",
    ENGLISH_MODEL_PREFIX + ".model"
)

1M English BPE tokenizer trained successfully.
Saved model: /content/drive/MyDrive/NNDL_English_Hindi_Translator/tokenizers/english_bpe_1m.model


In [4]:
# ============================================================
# STEP 4: LOAD CLEANED 300K TRAINING DATA
# ============================================================

train_df = pd.read_parquet(TRAIN_PATH)

print("Training dataset loaded successfully.")
print(f"Rows: {len(train_df):,}")
print("Columns:", train_df.columns.tolist())

display(train_df.head())

Training dataset loaded successfully.
Rows: 300,000
Columns: ['english', 'hindi']


,english,hindi
0,The judgment of the Bombay High Court referred...,"मुंबई उच्च न्यायालय का निर्णय, जो ऊपर अनुच्छेद..."
1,I am sure you all will do well in life.,मुझे विश्वास है कि आप सभी जीवन में अच्छी तरक्क...
2,"Balconies, cupolas and towers surmount the pal...","बालकनी, कूपोला और बड़ी बड़ी मीनारें इस महल को ..."
3,All the temples of Kerala invariably turn out ...,"केरल में सब मंदिर चाहे वह विष्णु मंदिर हों, शि..."
4,President of India Shri Pranab Mukherjee parti...,भारत के राष्ट्रपति श्री प्रणब मुखर्जी आज (30 न...


In [5]:
# ============================================================
# CHECK TOKENIZER TRAINING DATA SIZE
# ============================================================

print("train_df shape:", train_df.shape)

print("\nColumns:")
print(train_df.columns.tolist())

train_df shape: (300000, 2)

Columns:
['english', 'hindi']


In [ ]:
# ============================================================
# STEP 5: CREATE TEXT CORPORA FOR SENTENCEPIECE
# ============================================================

ENGLISH_CORPUS_PATH = os.path.join(
    TOKENIZER_DIR,
    "english_training_corpus.txt"
)

HINDI_CORPUS_PATH = os.path.join(
    TOKENIZER_DIR,
    "hindi_training_corpus.txt"
)

# Save one English sentence per line.
train_df["english"].to_csv(
    ENGLISH_CORPUS_PATH,
    index=False,
    header=False
)

# Save one Hindi sentence per line.
train_df["hindi"].to_csv(
    HINDI_CORPUS_PATH,
    index=False,
    header=False
)

print("Tokenizer training corpora created successfully.")

print("\nEnglish corpus:")
print(ENGLISH_CORPUS_PATH)

print("\nHindi corpus:")
print(HINDI_CORPUS_PATH)

Tokenizer training corpora created successfully.

English corpus:
/content/drive/MyDrive/NNDL_English_Hindi_Translator/tokenizers/english_training_corpus.txt

Hindi corpus:
/content/drive/MyDrive/NNDL_English_Hindi_Translator/tokenizers/hindi_training_corpus.txt


In [ ]:
# ============================================================
# STEP 6: TOKENIZER CONFIGURATION
# ============================================================

VOCAB_SIZE = 16000

# Special token IDs.
PAD_ID = 0
UNK_ID = 1
BOS_ID = 2
EOS_ID = 3

print("Vocabulary size:", VOCAB_SIZE)

print("\nSpecial token IDs:")
print("PAD:", PAD_ID)
print("UNK:", UNK_ID)
print("BOS:", BOS_ID)
print("EOS:", EOS_ID)

Vocabulary size: 16000

Special token IDs:
PAD: 0
UNK: 1
BOS: 2
EOS: 3


In [12]:
# ============================================================
# STEP 7: TRAIN ENGLISH SENTENCEPIECE BPE TOKENIZER
# ============================================================

ENGLISH_MODEL_PREFIX = os.path.join(
    TOKENIZER_DIR,
    "english_bpe"
)

spm.SentencePieceTrainer.train(
    input=ENGLISH_CORPUS_PATH,
    model_prefix=ENGLISH_MODEL_PREFIX,

    # BPE = Byte Pair Encoding
    model_type="bpe",

    vocab_size=VOCAB_SIZE,

    # Keep all characters found in our corpus.
    character_coverage=1.0,

    # Fixed special token IDs.
    pad_id=PAD_ID,
    unk_id=UNK_ID,
    bos_id=BOS_ID,
    eos_id=EOS_ID,

    pad_piece="<pad>",
    unk_piece="<unk>",
    bos_piece="<s>",
    eos_piece="</s>"
)

print("English BPE tokenizer trained successfully.")

English BPE tokenizer trained successfully.


In [13]:
# ============================================================
# STEP 8: TRAIN HINDI SENTENCEPIECE BPE TOKENIZER - 1M DATA
# ============================================================

HINDI_MODEL_PREFIX = os.path.join(
    TOKENIZER_DIR,
    "hindi_bpe_1m"
)

spm.SentencePieceTrainer.train(
    input=HINDI_CORPUS_PATH,
    model_prefix=HINDI_MODEL_PREFIX,

    # BPE tokenizer
    model_type="bpe",

    # Keep the same vocabulary size
    vocab_size=VOCAB_SIZE,

    character_coverage=1.0,

    # Fixed special token IDs
    pad_id=PAD_ID,
    unk_id=UNK_ID,
    bos_id=BOS_ID,
    eos_id=EOS_ID,

    pad_piece="<pad>",
    unk_piece="<unk>",
    bos_piece="<s>",
    eos_piece="</s>"
)

print("1M Hindi BPE tokenizer trained successfully.")
print(
    "Saved model:",
    HINDI_MODEL_PREFIX + ".model"
)

1M Hindi BPE tokenizer trained successfully.
Saved model: /content/drive/MyDrive/NNDL_English_Hindi_Translator/tokenizers/hindi_bpe_1m.model


In [14]:
# ============================================================
# STEP 8: TRAIN HINDI SENTENCEPIECE BPE TOKENIZER - 1M DATA
# ============================================================

HINDI_MODEL_PREFIX = os.path.join(
    TOKENIZER_DIR,
    "hindi_bpe_1m"
)

spm.SentencePieceTrainer.train(
    input=HINDI_CORPUS_PATH,
    model_prefix=HINDI_MODEL_PREFIX,

    # BPE tokenizer
    model_type="bpe",

    # Keep the same vocabulary size
    vocab_size=VOCAB_SIZE,

    character_coverage=1.0,

    # Fixed special token IDs
    pad_id=PAD_ID,
    unk_id=UNK_ID,
    bos_id=BOS_ID,
    eos_id=EOS_ID,

    pad_piece="<pad>",
    unk_piece="<unk>",
    bos_piece="<s>",
    eos_piece="</s>"
)

print("1M Hindi BPE tokenizer trained successfully.")
print(
    "Saved model:",
    HINDI_MODEL_PREFIX + ".model"
)

1M Hindi BPE tokenizer trained successfully.
Saved model: /content/drive/MyDrive/NNDL_English_Hindi_Translator/tokenizers/hindi_bpe_1m.model


In [15]:
# ============================================================
# STEP 9: LOAD AND VERIFY NEW 1M TOKENIZERS
# ============================================================

english_tokenizer_1m = spm.SentencePieceProcessor(
    model_file=os.path.join(
        TOKENIZER_DIR,
        "english_bpe_1m.model"
    )
)

hindi_tokenizer_1m = spm.SentencePieceProcessor(
    model_file=os.path.join(
        TOKENIZER_DIR,
        "hindi_bpe_1m.model"
    )
)

print("English tokenizer vocab size:",
      english_tokenizer_1m.get_piece_size())

print("Hindi tokenizer vocab size:",
      hindi_tokenizer_1m.get_piece_size())

print("\nEnglish special IDs:")
print("PAD:", english_tokenizer_1m.pad_id())
print("UNK:", english_tokenizer_1m.unk_id())
print("BOS:", english_tokenizer_1m.bos_id())
print("EOS:", english_tokenizer_1m.eos_id())

print("\nHindi special IDs:")
print("PAD:", hindi_tokenizer_1m.pad_id())
print("UNK:", hindi_tokenizer_1m.unk_id())
print("BOS:", hindi_tokenizer_1m.bos_id())
print("EOS:", hindi_tokenizer_1m.eos_id())

English tokenizer vocab size: 16000
Hindi tokenizer vocab size: 16000

English special IDs:
PAD: 0
UNK: 1
BOS: 2
EOS: 3

Hindi special IDs:
PAD: 0
UNK: 1
BOS: 2
EOS: 3


In [ ]:
# ============================================================
# STEP 9: LOAD TRAINED TOKENIZERS
# ============================================================

english_tokenizer = spm.SentencePieceProcessor()

english_tokenizer.load(
    os.path.join(
        TOKENIZER_DIR,
        "english_bpe.model"
    )
)

hindi_tokenizer = spm.SentencePieceProcessor()

hindi_tokenizer.load(
    os.path.join(
        TOKENIZER_DIR,
        "hindi_bpe.model"
    )
)

print("Both tokenizers loaded successfully.")

print(
    "English vocabulary size:",
    english_tokenizer.get_piece_size()
)

print(
    "Hindi vocabulary size:",
    hindi_tokenizer.get_piece_size()
)

Both tokenizers loaded successfully.
English vocabulary size: 16000
Hindi vocabulary size: 16000


In [ ]:
# ============================================================
# STEP 10: VERIFY SPECIAL TOKEN IDs
# ============================================================

print("ENGLISH TOKENIZER")
print("-" * 35)

print("PAD:", english_tokenizer.pad_id())
print("UNK:", english_tokenizer.unk_id())
print("BOS:", english_tokenizer.bos_id())
print("EOS:", english_tokenizer.eos_id())


print("\nHINDI TOKENIZER")
print("-" * 35)

print("PAD:", hindi_tokenizer.pad_id())
print("UNK:", hindi_tokenizer.unk_id())
print("BOS:", hindi_tokenizer.bos_id())
print("EOS:", hindi_tokenizer.eos_id())

ENGLISH TOKENIZER
-----------------------------------
PAD: 0
UNK: 1
BOS: 2
EOS: 3

HINDI TOKENIZER
-----------------------------------
PAD: 0
UNK: 1
BOS: 2
EOS: 3


In [ ]:
# ============================================================
# STEP 11: TEST ENGLISH TOKENIZER
# ============================================================

english_example = "I am learning deep learning."

english_pieces = english_tokenizer.encode(
    english_example,
    out_type=str
)

english_ids = english_tokenizer.encode(
    english_example,
    out_type=int
)

print("Original sentence:")
print(english_example)

print("\nBPE pieces:")
print(english_pieces)

print("\nToken IDs:")
print(english_ids)

Original sentence:
I am learning deep learning.

BPE pieces:
['▁I', '▁am', '▁learning', '▁deep', '▁learning', '.']

Token IDs:
[62, 337, 2966, 2316, 2966, 15686]


In [ ]:
# ============================================================
# STEP 12: TEST ENGLISH DECODING
# ============================================================

decoded_english = english_tokenizer.decode(
    english_ids
)

print("Original:")
print(english_example)

print("\nDecoded:")
print(decoded_english)

Original:
I am learning deep learning.

Decoded:
I am learning deep learning.


In [ ]:
# ============================================================
# STEP 13: TEST HINDI TOKENIZER
# ============================================================

hindi_example = "मुझे भारत पसंद है।"

hindi_pieces = hindi_tokenizer.encode(
    hindi_example,
    out_type=str
)

hindi_ids = hindi_tokenizer.encode(
    hindi_example,
    out_type=int
)

print("Original sentence:")
print(hindi_example)

print("\nBPE pieces:")
print(hindi_pieces)

print("\nToken IDs:")
print(hindi_ids)

Original sentence:
मुझे भारत पसंद है।

BPE pieces:
['▁मुझे', '▁भारत', '▁पसंद', '▁है', '।']

Token IDs:
[558, 276, 2884, 11, 15660]


In [ ]:
# ============================================================
# STEP 14: TEST HINDI DECODING
# ============================================================

decoded_hindi = hindi_tokenizer.decode(
    hindi_ids
)

print("Original:")
print(hindi_example)

print("\nDecoded:")
print(decoded_hindi)

Original:
मुझे भारत पसंद है।

Decoded:
मुझे भारत पसंद है।


In [ ]:
# ============================================================
# STEP 15: CALCULATE ENGLISH BPE TOKEN LENGTHS
# ============================================================

train_df["english_bpe_length"] = train_df["english"].apply(
    lambda text: len(
        english_tokenizer.encode(
            text,
            out_type=int
        )
    )
)

print("English BPE token lengths calculated.")

English BPE token lengths calculated.


In [ ]:
# ============================================================
# STEP 16: CALCULATE HINDI BPE TOKEN LENGTHS
# ============================================================

train_df["hindi_bpe_length"] = train_df["hindi"].apply(
    lambda text: len(
        hindi_tokenizer.encode(
            text,
            out_type=int
        )
    )
)

print("Hindi BPE token lengths calculated.")

Hindi BPE token lengths calculated.


In [ ]:
# ============================================================
# STEP 17: BPE TOKEN LENGTH STATISTICS
# ============================================================

print("ENGLISH BPE TOKEN LENGTH")
print("-" * 45)

print(
    train_df["english_bpe_length"].describe(
        percentiles=[
            0.50,
            0.90,
            0.95,
            0.99
        ]
    )
)


print("\nHINDI BPE TOKEN LENGTH")
print("-" * 45)

print(
    train_df["hindi_bpe_length"].describe(
        percentiles=[
            0.50,
            0.90,
            0.95,
            0.99
        ]
    )
)

ENGLISH BPE TOKEN LENGTH
---------------------------------------------
count    300000.000000
mean         15.890133
std          13.360670
min           1.000000
50%          13.000000
90%          35.000000
95%          42.000000
99%          56.000000
max         167.000000
Name: english_bpe_length, dtype: float64

HINDI BPE TOKEN LENGTH
---------------------------------------------
count    300000.000000
mean         16.996537
std          14.352325
min           1.000000
50%          13.000000
90%          38.000000
95%          46.000000
99%          59.000000
max         331.000000
Name: hindi_bpe_length, dtype: float64


In [ ]:
# ============================================================
# STEP 18: CHECK COVERAGE FOR MAX SEQUENCE LENGTH = 64
# ============================================================

# Two positions are reserved for:
# BOS = beginning of sentence
# EOS = end of sentence
MAX_SEQUENCE_LENGTH = 64
MAX_CONTENT_TOKENS = MAX_SEQUENCE_LENGTH - 2

english_within_limit = (
    train_df["english_bpe_length"] <= MAX_CONTENT_TOKENS
).mean() * 100

hindi_within_limit = (
    train_df["hindi_bpe_length"] <= MAX_CONTENT_TOKENS
).mean() * 100

both_within_limit = (
    (
        train_df["english_bpe_length"] <= MAX_CONTENT_TOKENS
    )
    &
    (
        train_df["hindi_bpe_length"] <= MAX_CONTENT_TOKENS
    )
).mean() * 100


print("MAX SEQUENCE LENGTH:", MAX_SEQUENCE_LENGTH)
print("Maximum content tokens:", MAX_CONTENT_TOKENS)

print(
    f"\nEnglish sentences within limit: "
    f"{english_within_limit:.2f}%"
)

print(
    f"Hindi sentences within limit: "
    f"{hindi_within_limit:.2f}%"
)

print(
    f"Pairs with both sides within limit: "
    f"{both_within_limit:.2f}%"
)

MAX SEQUENCE LENGTH: 64
Maximum content tokens: 62

English sentences within limit: 99.62%
Hindi sentences within limit: 99.44%
Pairs with both sides within limit: 99.21%


In [ ]:
# ============================================================
# STEP 19: SAVE TOKENIZER LENGTH STATISTICS
# ============================================================

import pandas as pd

tokenizer_stats = pd.DataFrame({
    "metric": [
        "english_mean",
        "english_median",
        "english_p90",
        "english_p95",
        "english_p99",
        "english_max",
        "hindi_mean",
        "hindi_median",
        "hindi_p90",
        "hindi_p95",
        "hindi_p99",
        "hindi_max"
    ],

    "value": [
        train_df["english_bpe_length"].mean(),
        train_df["english_bpe_length"].median(),
        train_df["english_bpe_length"].quantile(0.90),
        train_df["english_bpe_length"].quantile(0.95),
        train_df["english_bpe_length"].quantile(0.99),
        train_df["english_bpe_length"].max(),

        train_df["hindi_bpe_length"].mean(),
        train_df["hindi_bpe_length"].median(),
        train_df["hindi_bpe_length"].quantile(0.90),
        train_df["hindi_bpe_length"].quantile(0.95),
        train_df["hindi_bpe_length"].quantile(0.99),
        train_df["hindi_bpe_length"].max()
    ]
})

TOKEN_STATS_PATH = os.path.join(
    PROJECT_ROOT,
    "results",
    "metrics",
    "tokenizer_length_statistics.csv"
)

tokenizer_stats.to_csv(
    TOKEN_STATS_PATH,
    index=False
)

print("Tokenizer statistics saved successfully.")
print(TOKEN_STATS_PATH)

Tokenizer statistics saved successfully.
/content/drive/MyDrive/NNDL_English_Hindi_Translator/results/metrics/tokenizer_length_statistics.csv


In [ ]:
# ============================================================
# STEP 20: SAVE TOKENIZER / SEQUENCE CONFIGURATION
# ============================================================

import json

config = {
    "training_pairs": 300000,
    "random_seed": 42,

    "english_vocab_size": 16000,
    "hindi_vocab_size": 16000,

    "pad_id": 0,
    "unk_id": 1,
    "bos_id": 2,
    "eos_id": 3,

    "max_sequence_length": 64,
    "max_content_tokens": 62,

    "training_word_length_limit": 50,

    "english_tokenizer": "english_bpe.model",
    "hindi_tokenizer": "hindi_bpe.model"
}


CONFIG_PATH = os.path.join(
    TOKENIZER_DIR,
    "tokenizer_config.json"
)

with open(
    CONFIG_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        config,
        file,
        indent=4,
        ensure_ascii=False
    )


print("Tokenizer configuration saved successfully.")
print(CONFIG_PATH)

Tokenizer configuration saved successfully.
/content/drive/MyDrive/NNDL_English_Hindi_Translator/tokenizers/tokenizer_config.json
